### LLM Archi Overview

In [62]:
GPT_config_124M={
    "vocab_size":50257,
    "context_length":1024,
    "emb_dim":768,
    "n_heads":12,
    "n_layers":12,
    "drop_rate":0.1,
    "qkv_bias":False
}

In [63]:
import torch
import torch.nn as nn 

In [64]:
class DummuGPT(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.tok_emb=nn.Embedding(cfg["vocab_size"],cfg["emb_dim"])
        self.pos_emb=nn.Embedding(cfg["context_length"],cfg["emb_dim"])
        self.drop_emb=nn.Dropout(cfg["drop_rate"])

        ## use plcaeholder of transformer clas
        self.trf_blocks=nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        ## use placeholder of noraml layer 
        self.final_norm=DummyLayerNorm(cfg['emb_dim'])
        self.out_head=nn.Linear(
            cfg['emb_dim'],cfg['vocab_size'],bias=False
        )

    def forward(self,in_idx):
        batch_size,seq_len=in_idx.shape 
        tok_embeds=self.tok_emb(in_idx)
        pos_embeds=self.pos_emb(torch.arange(seq_len,device=in_idx.device))
        x=tok_embeds+pos_embeds
        x=self.drop_emb(x)
        x=self.trf_blocks(x)
        x=self.final_norm(x)
        logits=self.out_head(x)
        return logits


class DummyTransformerBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
    
    def forward(self,x):
        return x

class DummyLayerNorm(nn.Module):
    def __init__(self,normalized_shape,eps=1e-5):
        super().__init__()

    def forward(self,x):
        return x

    

In [65]:
import tiktoken
tokenizer=tiktoken.get_encoding("gpt2")
batch=[]
txt1="how are you"
txt2="i am good"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch

[tensor([4919,  389,  345]), tensor([ 72, 716, 922])]

In [66]:
batch=torch.stack(batch,dim=0)
batch

tensor([[4919,  389,  345],
        [  72,  716,  922]])

In [67]:
torch.manual_seed(123)
model=DummuGPT(GPT_config_124M)
logits=model(batch)
logits

tensor([[[ 0.5703,  0.3038, -1.4865,  ..., -0.3215,  0.0581,  0.5639],
         [-0.1355, -0.4994,  0.0919,  ..., -0.0895,  0.6472,  0.2599],
         [ 0.2963,  1.8822, -0.2460,  ...,  0.0705,  0.6123, -1.1480]],

        [[-0.3776, -0.0847, -0.2463,  ..., -0.3162, -0.0799, -0.3225],
         [-0.9311,  1.5354, -0.2682,  ..., -0.8793, -0.0554,  1.3383],
         [ 0.8358, -0.0880, -0.5626,  ...,  1.9181,  0.2339, -0.9430]]],
       grad_fn=<UnsafeViewBackward0>)

In [68]:
logits.shape

torch.Size([2, 3, 50257])

In [71]:
torch.max(logits,dim=-1)

torch.return_types.max(
values=tensor([[3.5674, 3.4527, 3.6344],
        [3.1623, 3.3666, 3.5392]], grad_fn=<MaxBackward0>),
indices=tensor([[24520, 15572,  1341],
        [12287, 12210, 18161]]))

In [73]:
torch.max(logits,dim=1)

torch.return_types.max(
values=tensor([[ 0.5703,  1.8822,  0.0919,  ...,  0.0705,  0.6472,  0.5639],
        [ 0.8358,  1.5354, -0.2463,  ...,  1.9181,  0.2339,  1.3383]],
       grad_fn=<MaxBackward0>),
indices=tensor([[0, 2, 1,  ..., 2, 1, 0],
        [2, 1, 0,  ..., 2, 2, 1]]))